In [2]:
import time
import urllib.request
import re
import requests
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from selenium.common.exceptions import TimeoutException


In [ ]:
def yes24_crawl(max_page):
    search_list = []
    title_list = []
    name_list = []
    price_list = []
    com_list = []
    date_list = []
    image_list = []

    folder_path = r"imgaes\yes24"

    url = "https://www.yes24.com/Main/default.aspx"

    driver = webdriver.Chrome()
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)

    driver.get(url)
    ban = True
    need_input = True

    while ban:
        if need_input:
            keyword = input("검색어 입력: ")
            need_input = False

        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.ID, "query"))
            )

            search_box.clear()
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)

            try:
                WebDriverWait(driver, 3).until(EC.alert_is_present())
                alert = driver.switch_to.alert
                alert.accept()
                    
                time.sleep(2)
                need_input = True
                continue 
                    
            except TimeoutException:
                pass
            ban = False
        
        except Exception as e:
            continue
    # ----------------------------------------------------- 검색

    for page in range(1, max_page + 1): 
            
            if page == max_page + 1:                      
                print(f"크롤링 끝")
                break
            

            print(f"{page}번 페이지 크롤링")

            req = driver.page_source

            soup = BeautifulSoup(req, "html.parser")
            
            ALL_books = soup.find("ul", id = "yesSchList").find_all("li")

            for row in ALL_books:
                if not row.select_one(".gd_name"): 
                    continue
                
                title = row.select_one(".gd_name").text 
                name = row.select_one(".info_auth").text if row.select_one(".info_auth") else ""
                price = row.select_one(".yes_b").text if row.select_one(".yes_b") else ""
                com = row.select_one(".info_pub").text if row.select_one(".info_pub") else ""
                ddatee = row.select_one(".info_date").text.strip() if row.select_one(".info_date") else ""
                img_tag = row.select_one(".img_bdr img")
                

                if img_tag:
                    img_url = img_tag.get("data-original") or img_tag.get("src")
                    
                    if img_url:
                        try: 
                            img_safe_title = re.sub(r'[\/:*?"<>|]', '', title) 
                            img_save = os.path.join(folder_path, f"{img_safe_title}.jpg")
                            urllib.request.urlretrieve(img_url, img_save)
                        except:
                            print(f"이미지 다운로드 실패 ({title}): {e}")
                            img_save = "다운로드 실패"
                            

                        
                search_list.append(keyword)
                title_list.append(title)
                name_list.append(name)
                price_list.append(price)
                com_list.append(com)
                date_list.append(ddatee)
                image_list.append(img_save)
            
                

            if page < max_page:
                next_start_index = page + 1
                try:
                    next_btn = wait.until(
                        EC.element_to_be_clickable((By.LINK_TEXT, str(next_start_index)))
                    )
                    next_btn.click()
                    time.sleep(3)
                except Exception as e:
                    print("이동할 페이지가 없어서 종료", e)
                    break
            else:
                print("목표 페이지에 도달")
            
    df = pd.DataFrame({
        "검색어": search_list,
        "도서명": title_list,
        "저자": name_list,
        "가격": price_list,
        "출판사": com_list,
        "출판일": date_list,
        "이미지 저장 경로": image_list
    })

    driver.close()
    return df

yes24_crawl(3)



1번 페이지 크롤링
2번 페이지 크롤링
3번 페이지 크롤링
목표 페이지에 도달하여 크롤링을 종료합니다.


,검색어,도서명,저자,가격,출판사,출판일,이미지 저장 경로
0,123,"2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상",\n최태성 저\n,"17,100",이투스북,2025년 12월,imgaes\yes24\2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(...
1,123,"2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하",\n최태성 저\n,"16,650",이투스북,2025년 12월,imgaes\yes24\2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(...
2,123,"2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상+하 세트",\n최태성 저\n,"33,750",이투스북,2025년 12월,imgaes\yes24\2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(...
3,123,"2026 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심화(1,2,3급)",\n최태성 저\n,"19,800",이투스북,2025년 12월,imgaes\yes24\2026 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력...
4,123,2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화 세트,\n최태성 저\n,"53,550",이투스북,2025년 12월,imgaes\yes24\2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화 ...
...,...,...,...,...,...,...,...
67,123,처음공부 123 : 4권 모양·색깔·부분과 전체·관련 그림,"\n강동희, 박연지, 이정희, 전혜진 글\n ...","21,600",모듀efe,2025년 12월,imgaes\yes24\처음공부 123 4권 모양·색깔·부분과 전체·관련 그림.jpg
68,123,미니 깜찍 팝업북 123 세트,"\n조현경, 노란 글 / 삼식이, 이수희 그림\n ...","40,950",꿈꾸는달팽이,2023년 04월,imgaes\yes24\미니 깜찍 팝업북 123 세트.jpg
69,123,터닝메카드 스티커놀이 123,\n 편집부 저\n ...,"6,750",드림박스,2015년 11월,imgaes\yes24\터닝메카드 스티커놀이 123.jpg
70,123,123미니쌤의 초등 수학 로드맵,\n김민희 저\n,"13,000",생각지도,2021년 12월,imgaes\yes24\123미니쌤의 초등 수학 로드맵.jpg


In [ ]:
def kyobo_crawl(max_page):
    search_list = []
    title_list = []
    name_list = []
    price_list = []
    com_list = []
    date_list = []
    image_list = []

    folder_path = r"imgaes\kyobo"

    url = "https://product.kyobobook.co.kr"

    driver = webdriver.Chrome()
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)

    driver.get(url)

    while True:
        try:
            search_box = wait.until(EC.presence_of_element_located((By.ID, "searchKeyword")))
            keyword = input("검색어 입력: ")
            
            search_box.clear()
            time.sleep(3)
            for i in keyword:
                search_box.send_keys(i)
                time.sleep(0.5)
                
            search_box.send_keys(Keys.ENTER)

            try:
                confirm_btn = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), '확인')]"))
                )
                confirm_btn.click()
                print("팝업 발생: 다시 입력하세요.")
                continue 
            
            except TimeoutException: # 검색 되면 끝낼거임
                break

        except Exception as e:
            print(f"오류 발생: {e}")
            driver.refresh()
            time.sleep(3)
            continue
        
    #---------------------------------------------------------------------- 사이트 열고 검색

    for page in range(1, max_page + 1):
            print(f"{page}번 페이지 크롤링")

            req = driver.page_source

            soup = BeautifulSoup(req, "html.parser")

            time.sleep(2)
            
            ALL_books = soup.find("ul", class_="prod_list").find_all("li")
            

            for row in ALL_books:
                if not row.select_one("a.prod_info"): 
                    continue
                
                title = row.select_one("a.prod_info").text 
                name = row.select_one("div.auto_overflow_wrap.prod_author_group").text if row.select_one("div.auto_overflow_inner") else ""
                name = name.replace("더보기", "")
                price = row.select_one(".val").text if row.select_one(".val") else ""
                com = row.select_one(".prod_publish").text if row.select_one(".prod_publish") else ""
                ddatee = row.select_one(".date").text.strip() if row.select_one(".date") else ""
                img_tag = row.select_one(".img_box img")


                title = re.sub(r'\s+', ' ', title).strip()
                name = re.sub(r'\s+', ' ', name).strip()
                price = re.sub(r'\s+', ' ', price).strip()
                com = re.sub(r'\s+', ' ', com).strip()
                ddatee = re.sub(r'\s+', ' ', ddatee).strip()
                

                if img_tag:
                    img_url = img_tag.get("data-original") or img_tag.get("src")
                    
                    if img_url:
                        try: 
                            img_safe_title = re.sub(r'[\\/:*?"<>|\n\r]', '', title).strip()
                            img_save = os.path.join(folder_path, f"{img_safe_title}.jpg")
                            urllib.request.urlretrieve(img_url, img_save)
                        except Exception as e:
                            print(f"이미지 다운로드 실패 ({title}): {e}")
                            img_save = "다운로드 실패"
                            

                        
                search_list.append(keyword)
                title_list.append(title)
                name_list.append(name)
                price_list.append(price)
                com_list.append(com)
                date_list.append(ddatee)
                image_list.append(img_save)
            
                next_start_index = page + 1

            try:
                next_btn = wait.until(
                EC.element_to_be_clickable((By.LINK_TEXT, str(next_start_index)))
                
                )
                next_btn.click()
                time.sleep(5)

            except Exception as e:
                print("이동할 페이지 없어서 종료", e)
                break
            
    df = pd.DataFrame({
        "검색어": search_list,
        "도서명": title_list,
        "저자": name_list,
        "가격": price_list,
        "출판사": com_list,
        "출판일": date_list,
        "이미지 저장 경로": image_list
    })

    driver.close()
    return df


kyobo_crawl(3)

1번 페이지 크롤링
2번 페이지 크롤링
3번 페이지 크롤링


,검색어,도서명,저자,가격,출판사,출판일,이미지 저장 경로
0,123,"[국내도서] 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급...",최태성 저자(글),"17,100",이투스북 · 2025년 12월 09일,2025년 12월 09일,imgaes\kyobo\[국내도서] 2026 큰별쌤 최태성의 별별한국사 한국사능력검...
1,123,"[국내도서] 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급...",최태성 저자(글),"16,650",이투스북 · 2025년 12월 09일,2025년 12월 09일,imgaes\kyobo\[국내도서] 2026 큰별쌤 최태성의 별별한국사 한국사능력검...
2,123,[국내도서] 2026 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심...,최태성 저자(글),"19,800",이투스북 · 2025년 12월 16일,2025년 12월 16일,imgaes\kyobo\[국내도서] 2026 큰별쌤 최태성의 별별한국사 기출 500...
3,123,[국내도서] 큰별쌤 최태성의 별별한국사 시대별 기출문제집 한국사능력검정시험 심화(1...,최태성 저자(글),"17,730",이투스북 · 2025년 06월 11일,2025년 06월 11일,imgaes\kyobo\[국내도서] 큰별쌤 최태성의 별별한국사 시대별 기출문제집 한...
4,123,[국내도서] Wipe-Clean 넘버블록스 썼다 지웠다 123,펭귄랜덤하우스코리아 편집부 저자(글),"10,800",펭귄랜덤하우스코리아 · 2026년 04월 20일,2026년 04월 20일,imgaes\kyobo\[국내도서] Wipe-Clean 넘버블록스 썼다 지웠다 12...
5,123,[국내도서] 수놀이 그림책 잘잘잘 123,이억배 저자(글),"10,800",사계절 · 2008년 03월 20일,2008년 03월 20일,imgaes\kyobo\[국내도서] 수놀이 그림책 잘잘잘 123.jpg
6,123,[국내도서] My Little Tiger 수학 연습장: 123,삼성출판사 편집부 저자(글),"3,600",삼성출판사 · 2022년 10월 01일,2022년 10월 01일,imgaes\kyobo\[국내도서] My Little Tiger 수학 연습장 123...
7,123,[국내도서] 맨발걷기 동의보감,박동창 저자(글),"7,920",국일미디어 · 2024년 09월 20일,2024년 09월 20일,imgaes\kyobo\[국내도서] 맨발걷기 동의보감.jpg
8,123,[국내도서] 사랑은123,밤코 저자(글),"12,600",바둑이하우스 · 2023년 05월 01일,2023년 05월 01일,imgaes\kyobo\[국내도서] 사랑은123.jpg
9,123,[국내도서] 쉽게 배우는 코바늘 손뜨개 무늬 123,일본 보그사 저자(글) | 배혜영 번역 | 김영희 감수 외,"13,500",한스미디어 · 2013년 02월 22일,2013년 02월 22일,imgaes\kyobo\[국내도서] 쉽게 배우는 코바늘 손뜨개 무늬 123.jpg


In [ ]:
def aladin_crawl():
    search_list = []
    title_list = []
    name_list = []
    price_list = []
    com_list = []
    date_list = []
    image_list = []

    folder_path = r"imgaes\aladin"

    url = "https://www.aladin.co.kr/home/welcome.aspx?NaPm=ct%3dmr0aiy2v%7cci%3dcheckout%7ctr%3dds%7ctrx%3dnull%7chk%3df3c4b01d00657ef931dca9c3e7b5dbee5eb6fdb3"

    driver = webdriver.Chrome()
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)

    driver.get(url)
    ban = True
    need_input = True

    while ban:
        if need_input:
            keyword = input("검색어 입력: ")
            need_input = False

        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.ID, "SearchWord"))
            )

            search_box.clear()
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)

            try:
                WebDriverWait(driver, 3).until(EC.alert_is_present())
                alert = driver.switch_to.alert
                alert.accept()
                    
                time.sleep(2)
                need_input = True
                continue 
                    
            except TimeoutException:
                pass
            ban = False
        
        except Exception as e:
            continue
    # ----------------------------------------------------- 검색

    max_page = 3
    for page in range(1, max_page):   #  1, max_page + 1 
            
            if page == max_page + 1:                      
                print(f"크롤링 끝")
                break
            

            print(f"{page}번 페이지 크롤링")

            req = driver.page_source

            soup = BeautifulSoup(req, "html.parser")
            
            ALL_books = soup.select(".ss_bookbox")

            for row in ALL_books:
                if not row.select_one(".bo3"): 
                    continue
                
                title = row.select_one(".bo3").text #.
                # name = row.select_one(".info_auth").text if row.select_one(".info_auth") else ""
                price = row.select_one(".ss_p2").text if row.select_one(".ss_p2") else "" #.
                # com = row.select_one(".info_pub").text if row.select_one(".info_pub") else ""
                # ddatee = row.select_one(".info_date").text.strip() if row.select_one(".info_date") else ""
                

                for li in row.select("ul > li"):
                    if "|" in li.text:
                        ncd = li
                        break

                if ncd:
                    ncd = ncd.text.split('|')
                    
                    # 쪼개진 데이터가 3개 이상이라면 변수에 저장
                    if len(ncd) >= 3:
                        name = ncd[0].replace(',', '').strip() 
                        com = ncd[1].strip()                   
                        ddatee = ncd[2].strip()


                img_tag = row.select_one(".front_cover img") #.
                

                if img_tag:
                    img_url = img_tag.get("data-original") or img_tag.get("src")
                    
                    if img_url:
                        try: 
                            img_safe_title = re.sub(r'[\/:*?"<>|]', '', title) 
                            img_save = os.path.join(folder_path, f"{img_safe_title}.jpg")
                            urllib.request.urlretrieve(img_url, img_save)
                        except:
                            print(f"이미지 다운로드 실패 ({title}): {e}")
                            img_save = "다운로드 실패"
                            

                        
                search_list.append(keyword)
                title_list.append(title)
                name_list.append(name)
                price_list.append(price)
                com_list.append(com)
                date_list.append(ddatee)
                image_list.append(img_save)
            
            next_start_index = page + 1

            try:
                xpath_str = f"//div[@class='numbox']//a[text()='{next_start_index}']"
                next_btn = wait.until(
                    EC.element_to_be_clickable((By.XPATH, xpath_str))
                )
                next_btn.click()
                time.sleep(2)

            except Exception as e:
                print(f"이동할 {next_start_index}페이지가 없거나 에러 발생으로 종료:", e)
            
    df = pd.DataFrame({
        "검색어": search_list,
        "도서명": title_list,
        "저자": name_list,
        "가격": price_list,
        "출판사": com_list,
        "출판일": date_list,
        "이미지 저장 경로": image_list
    })

    df.to_excel("알라딘.xlsx", index = False)


    driver.close()
    return 
    
aladin_crawl()

1번 페이지 크롤링


UnboundLocalError: cannot access local variable 'next_start_index' where it is not associated with a value

In [ ]:
def crawl_start(yes24, kyobo, aladin):
    df01 = yes24_crawl(yes24)   
    df02 = kyobo_crawl(kyobo)
    df03 = aladin_crawl(aladin)

    # with pd.ExcelWriter("finential.xlsx") as writer:
    #     df01.to_excel(writer, sheet_name="yes24 시트", index=False)
    #     df02.to_excel(writer, sheet_name="kyobo 시트", index=False)
    #     df03.to_excel(writer, sheet_name="aladin 시트", index=False)


In [ ]:
crawl_start(3, 2, 4)